# Basic Cleaning
Cleans the raw RTA dataset.

In [13]:
# imports
import pandas as pd
import numpy as np

In [14]:
# load raw data
df = pd.read_csv("../Dataset.csv")
df.head()

,Person Id,Time of Collision,Discharge Outcome,Facilities for Daily Activity,Access To Wash Room,Type of Toilet Modification,Bystander Expenditure per day,Traveling Expenditure per day,Total Traveling Expenditure For Whole Hospital Stay,Family Monthly Income Before Accident,...,Management Done At First Hospital,Management Name 1,Management Name 2,Number of Wards Stayed,Reason for stay in first ward,Number of days in first ward,Reason for stay in Second ward,Number of days in Second ward,Reason for stay in Third ward,Number of days in Third ward
0,85,00:00 - 03:00,Partial Recovery,Both Toilet and Bathing Facilities Available,No,No Modification done,Not Necessary,200-300,500-1000,15000-30000,...,Yes,Wound Cleaning And Dressing,Informed to JMO / Police,2,For observation,1,For specialist opinion,2,Not Necessary,Not Necessary
1,95,00:00 - 03:00,Complete Recovery,Both Toilet and Bathing Facilities Available,Yes,No Modification done,More than 1000,200-300,Victim not willing to share/ Unable to respond...,30000-45000,...,Yes,Wound Cleaning And Dressing,Wound Suturing,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT
2,111,00:00 - 03:00,Partial Recovery,Both Toilet and Bathing Facilities Available,No,No Modification done,Not Necessary,400-500,1000-1500,45000-60000,...,Yes,Wound Cleaning And Dressing,Informed to JMO / Police,1,Three wheeler,1,Not Necessary,Not Necessary,Not Necessary,Not Necessary
3,127,00:00 - 03:00,Partial Recovery,Both Toilet and Bathing Facilities Available,Yes,No Modification done,Not Necessary,100-200,500-1000,30000-45000,...,Yes,Wound Cleaning And Dressing,Wound Suturing,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT
4,449,00:00 - 03:00,Complete Recovery,Both Toilet and Bathing Facilities Available,Yes,No Modification done,Not Necessary,0-100,0-500,Victim not willing to share/ Unable to respond...,...,No,Early Discharge/ Data Entry Staff Missed to E...,No Other Management,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to look the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT,Early Discharge/ Unable to Access the BHT


In [15]:
print(df.columns)

Index(['Person Id', 'Time of Collision', 'Discharge Outcome',
       'Facilities for Daily Activity', 'Access To Wash Room',
       'Type of Toilet Modification', 'Bystander Expenditure per day',
       'Traveling Expenditure per day',
       'Total Traveling Expenditure For Whole Hospital Stay',
       'Family Monthly Income Before Accident',
       'Family Monthly Income After Accident', 'Family Current Status',
       'Hospital Distance From Home', 'Any Insurance Claim Type',
       'Any Other Hospital Admission Expenditure', 'Ethnicity', 'Full Name',
       'Contact Number', 'Hospital Reference Number', 'Age', 'Date Of Birth',
       'Gender', 'Life Style', 'Educational Qualification', 'Address Street',
       'Occupation', 'Employment Type Name', 'Dress Name',
       'Mode Of Travel During Accident', 'Collision With ', 'Vehicle Insured',
       'Vehicle Insurance Type', 'Category of Road',
       'Incident At Time and Date', 'Collision Force From', 'Visiblity',
       'Road Condit

In [3]:
# drop identity / contact / address columns (PII, not useful for modeling)
pii_cols = ["Person Id", "Full Name", "Contact Number", "Address Street", "Ethnicity", "Hospital Reference Number"]
df = df.drop(columns=[c for c in pii_cols if c in df.columns])

In [4]:
# drop columns that only ever have one value (they carry no information)
constant_cols = [c for c in df.columns if not c.endswith("_missing_reason") and df[c].nunique(dropna=True) <= 1]
df = df.drop(columns=constant_cols)
constant_cols

['Facilities for Daily Activity', 'First Aid Given At Seen']

In [5]:
# strip stray spaces and standardize case on every text column, so the same value
# typed differently (extra spaces, different capitalization) isn't treated as two categories
text_cols = df.select_dtypes(include='object').columns

for c in text_cols:
    df[c] = df[c].astype(str).str.strip().str.lower()
    df.loc[df[c].isin(['nan', 'none']), c] = np.nan

In [6]:
# known placeholder phrases used instead of real NaN in this dataset
# (keys are lowercase - the normalization step above lowercases the data before this runs)
placeholder_map = {
    "victim not willing to share/ unable to respond/  early discharge": "patient_unable_unwilling",
    "victim doesn't have knowledge on distance/ not willing to share/ unable to respond/  early discharge" : "patient_has_no_knowledge",
    "victim unable to recall the time or early discharge": "patient_unable_unwilling",
    "victim not aware about the time/ not willing to share/ unable to respond/  early discharge": "patient_unable_unwilling",
    "early discharge/ unable to look the bht": "early_discharge",
    "early discharge/ unable to access the bht": "early_discharge",
    "data missing": "data_entry_gap",
    "missing data": "data_entry_gap",
    "not necessary": "not_applicable",
    "not necessary for student and unemployed" : "not_applicable",
    "na": "not_applicable",
    "unknown": "data_entry_gap"
}

In [7]:
# for every column, replace placeholder text with NaN and store the reason in a new column
for col in df.columns:
    reason_col = col + "_missing_reason"
    df[reason_col] = "not_missing"
    for phrase, reason in placeholder_map.items():
        mask = df[col] == phrase
        df.loc[mask, reason_col] = reason
        df.loc[mask, col] = np.nan
    # plain blank cells are also missing, just a data entry gap
    blank_mask = df[col].isna() & (df[reason_col] == "not_missing")
    df.loc[blank_mask, reason_col] = "data_entry_gap"

/tmp/ipykernel_43946/3613612513.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[reason_col] = "not_missing"
/tmp/ipykernel_43946/3613612513.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[reason_col] = "not_missing"
/tmp/ipykernel_43946/3613612513.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

In [8]:
# remove trailing "y" from age range values (e.g. "1-9y" -> "1-9")
df["Age"] = df["Age"].astype(str).str.replace("y", "", regex=False)

In [9]:
# save cleaned dataset for use in the model notebooks
df.to_csv("Preprocessed Dataset.csv", index=False)
df.shape

(6178, 132)

In [10]:
df.head()

,Time of Collision,Discharge Outcome,Access To Wash Room,Type of Toilet Modification,Bystander Expenditure per day,Traveling Expenditure per day,Total Traveling Expenditure For Whole Hospital Stay,Family Monthly Income Before Accident,Family Monthly Income After Accident,Family Current Status,...,Management Done At First Hospital_missing_reason,Management Name 1_missing_reason,Management Name 2_missing_reason,Number of Wards Stayed_missing_reason,Reason for stay in first ward_missing_reason,Number of days in first ward_missing_reason,Reason for stay in Second ward_missing_reason,Number of days in Second ward_missing_reason,Reason for stay in Third ward_missing_reason,Number of days in Third ward_missing_reason
0,00:00 - 03:00,partial recovery,no,no modification done,NaN,200-300,500-1000,15000-30000,25000-30000,mildly affected,...,not_missing,not_missing,not_missing,not_missing,not_missing,not_missing,not_missing,not_missing,not_applicable,not_applicable
1,00:00 - 03:00,complete recovery,yes,no modification done,more than 1000,200-300,NaN,30000-45000,25000-30000,mildly affected,...,not_missing,not_missing,not_missing,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge
2,00:00 - 03:00,partial recovery,no,no modification done,NaN,400-500,1000-1500,45000-60000,30000-40000,moderately affected,...,not_missing,not_missing,not_missing,not_missing,not_missing,not_missing,not_applicable,not_applicable,not_applicable,not_applicable
3,00:00 - 03:00,partial recovery,yes,no modification done,NaN,100-200,500-1000,30000-45000,10000-15000,moderately affected,...,not_missing,not_missing,not_missing,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge
4,00:00 - 03:00,complete recovery,yes,no modification done,NaN,0-100,0-500,NaN,NaN,mildly affected,...,not_missing,not_missing,not_missing,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge,early_discharge
